In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# ============================================================
# 1. Load Address CDC Work Table
# ============================================================

address_cdc_df = spark.table(
    "customer360_dev.control.address_cdc_work"
)

target_table = "customer360_dev.silver.addresses"

silver_delta = DeltaTable.forName(
    spark,
    target_table
)


# ============================================================
# 2. Count CDC Actions
# ============================================================

insert_count = (
    address_cdc_df
    .filter(F.col("cdc_action") == "INSERT")
    .count()
)

update_count = (
    address_cdc_df
    .filter(F.col("cdc_action") == "UPDATE")
    .count()
)

unchanged_count = (
    address_cdc_df
    .filter(F.col("cdc_action") == "UNCHANGED")
    .count()
)

print(f"INSERT    : {insert_count:,}")
print(f"UPDATE    : {update_count:,}")
print(f"UNCHANGED : {unchanged_count:,}")


# ============================================================
# 3. Capture Silver Count Before Merge
# ============================================================

silver_before_count = (
    spark.table(target_table).count()
)

print(
    f"Silver Addresses before merge: "
    f"{silver_before_count:,}"
)


# ============================================================
# 4. Keep Only INSERT / UPDATE Rows
# ============================================================

merge_source_df = (
    address_cdc_df
    .filter(
        F.col("cdc_action").isin(
            "INSERT",
            "UPDATE"
        )
    )
)


# ============================================================
# 5. Merge Into Silver Addresses
# ============================================================

(
    silver_delta.alias("tgt")

    .merge(
        merge_source_df.alias("src"),
        "tgt.address_id = src.address_id"
    )

    .whenMatchedUpdate(
        condition="""
            src.cdc_action = 'UPDATE'
            AND (
                tgt.updated_at IS NULL
                OR src.updated_at >= tgt.updated_at
            )
        """,

        set={
            "customer_id":
                "src.customer_id",

            "address_type":
                "src.address_type",

            "address_line1":
                "src.address_line1",

            "address_line2":
                "src.address_line2",

            "city":
                "src.city",

            "state":
                "src.state",

            "postal_code":
                "src.postal_code",

            "country":
                "src.country",

            "is_primary":
                "src.is_primary",

            "effective_from":
                "src.effective_from",

            "effective_to":
                "src.effective_to",

            "is_current":
                "src.is_current",

            "created_at":
                "src.created_at",

            "updated_at":
                "src.updated_at",

            "_source_file":
                "src._source_file",

            "_source_system":
                "src._source_system",

            "_load_type":
                "src._load_type",

            "_ingestion_timestamp":
                "src._ingestion_timestamp",

            "_silver_processed_at":
                "current_timestamp()"
        }
    )

    .whenNotMatchedInsert(
        condition="""
            src.cdc_action = 'INSERT'
        """,

        values={
            "address_id":
                "src.address_id",

            "customer_id":
                "src.customer_id",

            "address_type":
                "src.address_type",

            "address_line1":
                "src.address_line1",

            "address_line2":
                "src.address_line2",

            "city":
                "src.city",

            "state":
                "src.state",

            "postal_code":
                "src.postal_code",

            "country":
                "src.country",

            "is_primary":
                "src.is_primary",

            "effective_from":
                "src.effective_from",

            "effective_to":
                "src.effective_to",

            "is_current":
                "src.is_current",

            "created_at":
                "src.created_at",

            "updated_at":
                "src.updated_at",

            "_source_file":
                "src._source_file",

            "_source_system":
                "src._source_system",

            "_load_type":
                "src._load_type",

            "_ingestion_timestamp":
                "src._ingestion_timestamp",

            "_silver_processed_at":
                "current_timestamp()"
        }
    )

    .execute()
)


# ============================================================
# 6. Verify Silver Count After Merge
# ============================================================

silver_after_count = (
    spark.table(target_table).count()
)

actual_growth = (
    silver_after_count
    - silver_before_count
)

print(
    f"Silver before : {silver_before_count:,}"
)

print(
    f"Silver after  : {silver_after_count:,}"
)

print(
    f"Actual growth : {actual_growth:,}"
)

print(
    f"Expected inserts : {insert_count:,}"
)


# ============================================================
# 7. Verify Address ID Uniqueness
# ============================================================

duplicate_address_ids = (
    spark.table(target_table)
    .groupBy("address_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate address IDs in Silver:",
    duplicate_address_ids
)

INSERT    : 0
UPDATE    : 0
UNCHANGED : 0
Silver Addresses before merge: 1,178,400
Silver before : 1,178,400
Silver after  : 1,178,400
Actual growth : 0
Expected inserts : 0
Duplicate address IDs in Silver: 0
